# Lending Club loan data — 2016 only

Source: `lc_loan.csv`. Rows are restricted to `year == 2016` (loan issue year).

After loading, missing numeric values are filled with the **column mean**; extreme values are **IQR-capped** (see below). Use `df_clean` for modeling or plots that need cleaned data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "/Users/ebenblanks/Downloads/loan-return-prediction-hw-1-25-fall/lc_loan.csv"

df = pd.read_csv(DATA_PATH)
print(f"Full dataset: {len(df):,} rows")

df2016 = df[df["year"] == 2016].copy()
print(f"2016 only: {len(df2016):,} rows")

# Sanity check: issue dates should fall in 2016
df2016["issue_d"] = pd.to_datetime(df2016["issue_d"], format="%b-%Y")
assert df2016["issue_d"].dt.year.eq(2016).all(), "Unexpected issue year outside 2016"
del df  # free memory if desired
df2016.head()

Full dataset: 933,160 rows
2016 only: 316,955 rows


,id,loan_amnt,funded_amnt,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,total_acc,collections_12_mths_ex_med,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,year,early_default,return
616205,74716491,20000.0,20000.0,0.0975,643.00,B,B3,4 years,MORTGAGE,62000.0,...,36.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.153361
616206,74826201,7200.0,7200.0,0.0532,216.83,A,A1,10+ years,MORTGAGE,49000.0,...,36.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.003251
616207,74884818,32000.0,32000.0,0.0697,987.63,A,A3,10+ years,OWN,156000.0,...,13.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.104294
616208,75095270,17500.0,17500.0,0.1367,595.31,C,C3,2 years,RENT,46000.0,...,58.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.222260
616209,75124079,35000.0,35000.0,0.1953,1292.36,D,D5,10+ years,MORTGAGE,84000.0,...,30.0,0.0,0.0,0.0,0.0,1.0,1.0,2016,1,-0.814835


In [2]:
# Optional: save reduced dataset next to this notebook
OUT_CSV = "lc_loan_2016.csv"
df2016.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV} ({len(df2016):,} rows)")

Wrote lc_loan_2016.csv (316,955 rows)


## Schema and missing values

In [3]:
df2016.info()
print("\nMissing count (non-zero):")
miss = df2016.isna().sum()
print(miss[miss > 0].sort_values(ascending=False))

<class 'pandas.core.frame.DataFrame'>
Index: 316955 entries, 616205 to 933159
Data columns (total 37 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   id                          316955 non-null  int64         
 1   loan_amnt                   316955 non-null  float64       
 2   funded_amnt                 316955 non-null  float64       
 3   int_rate                    316955 non-null  float64       
 4   installment                 316955 non-null  float64       
 5   grade                       316955 non-null  object        
 6   sub_grade                   316955 non-null  object        
 7   emp_length                  294197 non-null  object        
 8   home_ownership              316955 non-null  object        
 9   annual_inc                  316955 non-null  float64       
 10  verification_status         316955 non-null  object        
 11  issue_d                     316955 non-

In [4]:
df2016.describe().T

,count,mean,min,25%,50%,75%,max,std
id,316955.0,82115046.419933,55716.0,74914168.5,81536806.0,90161273.0,96453160.0,8616606.268878
loan_amnt,316955.0,12675.998643,1000.0,6000.0,10000.0,17000.0,40000.0,8527.308461
funded_amnt,316955.0,12675.973403,1000.0,6000.0,10000.0,17000.0,40000.0,8527.304542
int_rate,316955.0,0.119215,0.0532,0.0859,0.1147,0.1399,0.3099,0.042532
installment,316955.0,420.701801,30.12,203.59,332.1,563.98,1584.9,286.372304
annual_inc,316955.0,77136.814304,4800.0,45000.0,65000.0,90703.5,9573072.0,74980.12789
issue_d,316955,2016-06-10 00:33:20.566641408,2016-01-01 00:00:00,2016-03-01 00:00:00,2016-06-01 00:00:00,2016-09-01 00:00:00,2016-12-01 00:00:00,NaN
dti,316955.0,18.354181,-1.0,11.88,17.78,24.45,49.96,8.614775
delinq_2yrs,316955.0,0.365279,0.0,0.0,0.0,0.0,22.0,0.956954
fico_range_low,316955.0,695.212507,660.0,670.0,685.0,710.0,845.0,31.791655


## Missing values (mean) and outliers (IQR cap)

- **Numeric:** any missing value is replaced with that column’s **mean** (computed on non-missing rows).
- **Outliers:** values outside \([Q_1 - 1.5 \times IQR,\, Q_3 + 1.5 \times IQR]\) are **winsorized** (clipped to the fence). Applied to numeric columns except `id`, `year`, and `return` (target left as-is).
- **`emp_length`** is categorical; it is still missing in some rows unless you impute separately (e.g. mode).

In [6]:
df_clean = df2016.copy()

num_cols = df_clean.select_dtypes(include=np.number).columns.tolist()

# Mean imputation for numeric columns
for col in num_cols:
    if df_clean[col].isna().any():
        m = df_clean[col].mean()
        n = df_clean[col].isna().sum()
        df_clean[col] = df_clean[col].fillna(m)
        print(f"Imputed {col}: {n:,} missing → mean = {m:.6g}")

# IQR winsorization (exclude id, year, return)
clip_exclude = {"id", "year", "return"}
clip_cols = [c for c in num_cols if c not in clip_exclude]

n_capped = {}
for col in clip_cols:
    s = df_clean[col]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    if not np.isfinite(iqr) or iqr == 0:
        continue
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    below = (s < lo).sum()
    above = (s > hi).sum()
    if below or above:
        n_capped[col] = int(below + above)
    df_clean[col] = s.clip(lower=lo, upper=hi)

print("\nIQR capped (row-wise value touches fence):", dict(sorted(n_capped.items(), key=lambda x: -x[1])[:15]))
print("…" if len(n_capped) > 15 else "")
print(f"\nRemaining missing (should be only non-numeric): {df_clean.isna().sum().sum()}")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

Imputed mths_since_last_delinq: 148,605 missing → mean = 33.4327

IQR capped (row-wise value touches fence): {'mths_since_last_delinq': 111667, 'revol_bal': 22249, 'annual_inc': 17176, 'loan_amnt': 12696, 'funded_amnt': 12696, 'inq_last_6mths': 12522, 'installment': 12414, 'open_acc': 12366, 'fico_range_low': 10554, 'fico_range_high': 10554, 'int_rate': 8163, 'total_acc': 7534, 'dti': 959, 'revol_util': 26}


Remaining missing (should be only non-numeric): 22758
emp_length    22758
dtype: int64


### Cleaned numeric summary

In [7]:
df_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
id,316955.0,82115046.419933,55716.0,74914168.5,81536806.0,90161273.0,96453160.0,8616606.268878
loan_amnt,316955.0,12583.968387,1000.0,6000.0,10000.0,17000.0,33500.0,8278.000536
funded_amnt,316955.0,12583.943147,1000.0,6000.0,10000.0,17000.0,33500.0,8277.996219
int_rate,316955.0,0.118574,0.0532,0.0859,0.1147,0.1399,0.2209,0.040652
installment,316955.0,416.190838,30.12,203.59,332.1,563.98,1104.565,273.977792
annual_inc,316955.0,72385.123182,4800.0,45000.0,65000.0,90703.5,159258.75,36434.794119
issue_d,316955,2016-06-10 00:33:20.566641408,2016-01-01 00:00:00,2016-03-01 00:00:00,2016-06-01 00:00:00,2016-09-01 00:00:00,2016-12-01 00:00:00,NaN
dti,316955.0,18.345199,-1.0,11.88,17.78,24.45,43.305,8.586535
delinq_2yrs,316955.0,0.365279,0.0,0.0,0.0,0.0,22.0,0.956954
fico_range_low,316955.0,694.421479,660.0,670.0,685.0,710.0,770.0,29.398709


In [8]:
# Optional: persist cleaned 2016 data
df_clean.to_csv("lc_loan_2016_clean.csv", index=False)
print(f"Wrote lc_loan_2016_clean.csv ({len(df_clean):,} rows)")

Wrote lc_loan_2016_clean.csv (316,955 rows)


In [10]:
# head on lc_loan_2016_clean.csv

df_clean

,id,loan_amnt,funded_amnt,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,total_acc,collections_12_mths_ex_med,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens,year,early_default,return
616205,74716491,20000.0,20000.0,0.0975,643.000,B,B3,4 years,MORTGAGE,62000.0,...,36.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.153361
616206,74826201,7200.0,7200.0,0.0532,216.830,A,A1,10+ years,MORTGAGE,49000.0,...,36.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.003251
616207,74884818,32000.0,32000.0,0.0697,987.630,A,A3,10+ years,OWN,156000.0,...,13.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.104294
616208,75095270,17500.0,17500.0,0.1367,595.310,C,C3,2 years,RENT,46000.0,...,52.5,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.222260
616209,75124079,33500.0,33500.0,0.1953,1104.565,D,D5,10+ years,MORTGAGE,84000.0,...,30.0,0.0,0.0,0.0,0.0,1.0,1.0,2016,1,-0.814835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
933155,90094364,4500.0,4500.0,0.1449,154.880,C,C4,10+ years,MORTGAGE,120000.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.127705
933156,89936581,4500.0,4500.0,0.0859,142.250,A,A5,1 year,MORTGAGE,30000.0,...,29.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.015690
933157,90006838,5000.0,5000.0,0.2149,189.640,D,D5,2 years,RENT,43000.0,...,19.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,0.059706
933158,89078718,2000.0,2000.0,0.0899,63.600,B,B1,10+ years,MORTGAGE,75000.0,...,25.0,0.0,0.0,0.0,0.0,0.0,0.0,2016,0,-0.531555
